# 04 — Predictive II: selection, tuning, and unlabelled structure

Mirroring `references/Day4_Team_Workbook.ipynb`: *one split lies, tuning beats guessing,
and not everything has a target.*


Reusable logic belongs in `src/startup_outcomes/`, not here — this notebook imports it and
defines nothing. Strip all outputs before committing, and check that it still runs under
**Restart & Run All**.

In [ ]:
# Activates the inline backend so a returned Figure renders. plots.py itself never
# imports pyplot, so no global state is created by the library.
%matplotlib inline

from startup_outcomes import features, plots
from startup_outcomes.load import load_raw
from startup_outcomes.models import card, protocol, supervised, unsupervised

df = load_raw(verify_checksum=True)
analysis = features.canonical_frame(df)
design, target = features.design_matrix(analysis)

## Step 1 — one split lies

The workbooks break the one-seed rule here deliberately, and the result is the argument
for why nothing else in this project uses a holdout: changing only the split seed moves
the score by more than every effect being measured.

In [ ]:
print(supervised.seed_instability(design, target))

## Step 2 — the model zoo, cross-validated

Scaling and one-hot encoding happen inside a `Pipeline`, so they are refit per fold. Fitting
a scaler on all rows first is the subtle leakage the workbooks warn about and then commit.

In [ ]:
board = protocol.leaderboard(supervised.leaderboard_results(design, target))
print(board[["features", "roc_auc", "pr_auc", "pr_auc_fold_sd"]].to_string())
plots.leaderboard_bars(board, base_rate=float(target.mean()))

Read the `pr_auc_fold_sd` column before the `pr_auc` column. Every gap in the top half of
that table is smaller than the spread beside it — including the gap between the
thirteen-feature model and the one-feature one.

## Step 3 — tuning, and what it buys

The grid is deliberately small, and `best_params_` is never reported: with eight near-tied
combinations, any tie-break change flips it.

In [ ]:
print(supervised.tuning(design, target))

## Step 4 — importance, read as blocks

The four size proxies correlate at r > 0.92 on logs, so whichever one a tree splits on
first is arbitrary. They are permuted together and reported as one group.

In [ ]:
print(supervised.grouped_importance(design, target).to_string())

## Step 5 — clustering: finding groups nobody labelled

On the one-hot matrix, k=2 scores a silhouette near 0.6 — an artifact of splitting on one
sparse binary. These run on the standardised continuous columns instead, where a distance
means something.

In [ ]:
matrix = protocol.log_numeric_matrix(design)
scan = unsupervised.cluster_scan(matrix)
print(scan.to_string())
plots.elbow_and_silhouette(scan)

A cluster is useless until you can name it. The only available name here is "big".

In [ ]:
best_k = unsupervised.chosen_k(matrix)
print(unsupervised.cluster_profile(analysis, matrix, best_k).to_string())
print()
print(unsupervised.cluster_separation(analysis, matrix, best_k))

## Step 6 — PCA: how much would a two-dimensional view throw away?

In [ ]:
curve = unsupervised.variance_curve(matrix)
print(curve.to_string())
print(unsupervised.pca_summary(matrix))
plots.variance_curve(curve)

The axes below are blends of logged, standardised columns. They mean nothing on their own
— never quote a component value to a reader.

In [ ]:
from sklearn.cluster import KMeans

from startup_outcomes.config import RANDOM_SEED

labels = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_SEED).fit_predict(matrix)
plots.component_scatter(
    unsupervised.pca_coordinates(matrix),
    labels,
    variance_shown_pct=float(unsupervised.pca_summary(matrix)["first_two_explained_pct"]),
)

## Step 7 — the model card

Rendered from a dict every value of which is pinned by a test, so it cannot drift from the
model it describes. Nothing is written to disk.

In [ ]:
print(card.model_card(analysis))

## Day 4 deliverable

We chose **gradient boosting** because it handles the categoricals natively and is the
most explainable of the top group — even though random forest scored 0.001 higher, which
is a sixth of the fold spread. The honest summary is that the choice does not matter,
because none of them beat one column.

## Scratch